# M1.1 · Modelos y agentes 🧠

**Lección oficial:** *Foundational models* · **Notebook original:** `module-1/1.1_foundational_models.ipynb`

## 🎯 Qué vas a aprender
- Qué es un **modelo** (LLM) y cómo hacerle una pregunta (`invoke`).
- Qué es la **temperatura**.
- Qué es un **agente** y en qué se diferencia de un modelo.
- Cómo leer la respuesta de un agente (la lista de `messages`).
- Qué es el **streaming** (ver la respuesta mientras se escribe).

## 🧸 Explicado sencillo
Imagina un **chef**:
- El **modelo** es el chef: sabe muchísimo, pero solo responde lo que le preguntas, una vez.
- El **agente** es el chef **dentro de una cocina**: puede repetir pasos, usar utensilios (*tools*) y seguir trabajando hasta terminar el plato.

En este notebook todavía no hay utensilios: primero conocemos al chef.

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · Crear el modelo

El curso original usa un modelo de OpenAI que cuesta dinero. Aquí usamos un modelo **gratis** que corre en nuestro servidor con **Ollama**.

- 🔸 **ORIGINAL DEL CURSO**: está comentado, no lo corras.
- 🟢 **ADAPTADO LOCAL**: esta es la que corres.

`get_model()` vive en `local_model.py` y por defecto usa **qwen3:8b**.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre (necesita API key de OpenAI, de pago)
# from langchain.chat_models import init_chat_model
# model = init_chat_model(model="gpt-5-nano")

# 🟢 ADAPTADO LOCAL
model = get_model()          # qwen3:8b en el servidor Ollama
print(model.model)

## 2 · Hacerle una pregunta: `invoke`

`invoke` = **preguntar y esperar** la respuesta completa.

La respuesta es un objeto `AIMessage`. Lo importante está en `.content` (el texto).

⏳ En CPU puede tardar entre 10 y 60 segundos. Es normal.

In [ ]:
response = model.invoke("What's the capital of the Moon?")
print(response.content)

### 🔍 ¿Qué más trae la respuesta?
`response_metadata` es como la **etiqueta** del mensaje: qué modelo respondió, cuánto tardó y cuántos *tokens* usó.

🧸 Un **token** es un pedacito de palabra. Los modelos leen y escriben en tokens, no en letras.

In [ ]:
from pprint import pprint
pprint(response.response_metadata)

🤔 **Pregunta para pensar:** la Luna no tiene capital. ¿El modelo lo dijo, o se inventó una? Cuando un modelo inventa algo con seguridad se llama **alucinación**.

## 3 · La temperatura 🎲

La **temperatura** controla qué tan "creativo" es el modelo:
- `0` → casi siempre responde lo mismo (bueno para datos y cálculos).
- `1` → más variado y creativo (bueno para cuentos).

🧠 **Mnemotecnia:** *temperatura baja = robot serio; temperatura alta = poeta.*

Corre la celda **dos veces** y compara.

In [ ]:
creativo = get_model(temperature=1.0)
print(creativo.invoke("Invent a name for the capital of the Moon. Only the name.").content)

## 4 · Tres formas de crear el mismo modelo

LangChain permite cambiar de proveedor (OpenAI, Anthropic, Google, Ollama…) sin reescribir el resto del código. Estas tres formas dan el mismo resultado:

| Forma | Cuándo se usa |
|---|---|
| `init_chat_model("ollama:qwen3:8b")` | Genérica: el texto `"proveedor:modelo"` |
| `ChatOllama(model="qwen3:8b")` | La clase directa del proveedor |
| `get_model()` | Nuestro atajo (usa `ChatOllama` por dentro) |

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_ollama import ChatOllama

url = os.getenv("OLLAMA_BASE_URL")
m1 = init_chat_model("ollama:qwen3:8b", base_url=url)
m2 = ChatOllama(model="qwen3:8b", base_url=url)
print(type(m1).__name__, "|", type(m2).__name__)   # las dos son ChatOllama

## 5 · Nuestro primer agente 🤖

`create_agent` envuelve al modelo en un **bucle**: el agente piensa, (si tuviera tools) las usa, y repite hasta tener la respuesta final.

Al agente no se le pasa un texto suelto: se le pasa un diccionario con una lista de **mensajes**:
```python
{"messages": [HumanMessage(content="...")]}
```

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent("gpt-5-nano")

# 🟢 ADAPTADO LOCAL
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=get_model())

response = agent.invoke({"messages": [HumanMessage(content="What's the capital of the Moon?")]})

### 🔍 Leer la respuesta del agente
El agente devuelve **toda la conversación**: `response["messages"]`.
- `[0]` es lo que tú preguntaste (🧑 Humano).
- `[-1]` (el último) es la respuesta final (🤖 Modelo).

`ver()` es una ayuda nuestra que la muestra con colores, más fácil que `pprint`.

In [ ]:
ver(response)
print("Solo la respuesta final:", response["messages"][-1].content)

## 6 · El historial manda 📜

El modelo **no recuerda nada por sí solo**: solo ve los mensajes que le mandas. Si le pasas una conversación "falsa", la cree.

Aquí le decimos que *él mismo* ya dijo que la capital es "Luna City".

In [ ]:
from langchain.messages import AIMessage

response = agent.invoke({"messages": [
    HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City"),
]})
ver(response)

🤔 **Pregunta para pensar:** ¿por qué ahora sí habla de Luna City como si existiera?

<details><summary>Ver respuesta</summary>

Porque el modelo trata el historial como verdad: "si yo lo dije antes, debe ser cierto". La memoria de un agente **es** la lista de mensajes; lo verás en la lección de memoria.
</details>

## 7 · Streaming: ver la respuesta mientras se escribe ⌨️

🧠 **Mnemotecnia:** `invoke` = *recibir la carta completa*; `stream` = *escuchar a alguien mientras habla*.

`stream_mode="messages"` entrega la respuesta **token por token**.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me about Luna City, the capital of the Moon, in 3 sentences")]},
    stream_mode="messages",
):
    if token.content:                        # algunos pedazos vienen vacíos
        print(token.content, end="", flush=True)

## ✍️ Tu turno
1. Cambia la pregunta de la sección 5 por una en **español** sobre algo que te guste.
2. Crea un agente con `get_model(temperature=1.0)` y pídele un nombre de mascota espacial. Córrelo 3 veces.
3. Usa `stream` con una pregunta larga y mira cómo aparece el texto.

> 💡 Si te atascas, pídele una **pista** al tutor (OpenCode). No le pidas la respuesta: pídele pistas.

## ✅ Mini quiz
1. ¿Qué diferencia hay entre un **modelo** y un **agente**?
2. ¿Dónde está el texto de la respuesta de un modelo?
3. ¿Qué temperatura usarías para un agente que hace cuentas?
4. En `response["messages"]`, ¿qué posición tiene la respuesta final?

<details><summary>Respuestas</summary>

1. El modelo responde una vez; el agente es un bucle que puede usar herramientas y repetir pasos.
2. En `response.content`.
3. Baja (0): queremos respuestas estables.
4. La última: `[-1]`.
</details>

## 🧠 Resumen en una línea
**Modelo = chef. Agente = chef con cocina. `invoke` = carta completa. `stream` = escuchar mientras habla.**